# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rimlazrek1/flyrank-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

## Setup (Local)

In [4]:
import json
import os
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd()
while not (ROOT / "data" / "raw").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ImportError:
    pass

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    import getpass
    HF_TOKEN = getpass.getpass("HF READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT_PREV = (
    "read_parquet(["
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet', "
    f"'{REL}/fact_content_daily_performance/month=2026-04/*.parquet'"
    "])"
)
FACT_NEXT = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-05/*.parquet')"

print("Connected.")
print("  FACT_PREV = month=2026-03 + 2026-04 (features)")
print("  FACT_NEXT = month=2026-05 (label)")


Connected.
  FACT_PREV = month=2026-03 + 2026-04 (features)
  FACT_NEXT = month=2026-05 (label)


## 1. Method choice and why

**Lane 4 — CTR opportunity scoring**

**Method:** Logistic Regression → rank by P(`is_ctr_underperformer`).

**Why:** we need a ranked list; logistic is simple and readable. Baseline is past `ctr_gap` (w04). Model may use more past signals (`imp_prev`, `pos_avg_prev`, `engagement_rate_prev`, `position_tier`) without using May / `ctr_gap` as features.

## 2. Split design

**Time split (built into the frame — same as w04)**  
Features = Mar–Apr 2026. Label = May 2026 underperformer. Decision moment = end of April.

**Client folds**  
`GroupKFold` by `client_hash_id` (4 folds). Train clients never appear in that fold’s test.

**Eval list (chosen)**  
- **One global list**, Precision@10 / @20 / @50 (headline @50).  
- **Every page = equal weight** (not “10 pages per client”).

**Not used here:** per-client lists / client-equal weight.

**Comparison contract (baseline = model):**  
1. same rows — eligible test pages (`imp_prev >= 500`)  
2. same folds — same client groups  
3. same metric + K — Precision@10 / @20 / @50  
4. same tie policy — score desc, then `imp_prev` desc  


## 3. Train + compare vs baseline

Folds → metrics JSON. Ranked queue = **client-holdout fold scores only** (each page scored when its client was held out). Same page `reason_code` / `action` as w04 (from past features, not the score). Time split is already in the frame (Mar–Apr → May). The playbook CSV is exported in `w07_action_playbook.ipynb`.

In [5]:
LABEL = "is_ctr_underperformer"
FEATURE_COLS = ["imp_prev", "ctr_prev", "pos_avg_prev", "engagement_rate_prev", "position_tier"]
NUM_COLS = ["imp_prev", "ctr_prev", "pos_avg_prev", "engagement_rate_prev"]
RANDOM_STATE = 42
IMP_FLOOR = 500
N_FOLDS = 4
KS = (10, 20, 50)


def precision_at_k(scores, labels, k, tie_break=None):
    """Rank by score desc, then tie_break desc (same contract for rule and model)."""
    scores = np.asarray(scores, dtype=float)
    labels = np.asarray(labels)
    if tie_break is None:
        order = np.argsort(-scores, kind="mergesort")
    else:
        tb = np.asarray(tie_break, dtype=float)
        order = np.lexsort((-tb, -scores))
    k = min(k, len(order))
    return float(labels[order[:k]].mean())


# --- Frame: Mar–Apr features, May label (w04 contract) ---
features = con.sql(f"""
    WITH prev_daily AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS imp_prev,
            SUM(gsc_clicks) AS clk_prev,
            AVG(NULLIF(gsc_avg_position, 0)) AS pos_avg_prev,
            SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END) AS sessions_prev,
            SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE 0 END) AS engaged_prev
        FROM {FACT_PREV}
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 500
    ),
    prev_scored AS (
        SELECT
            *,
            CASE WHEN imp_prev > 0 THEN 100.0 * clk_prev / imp_prev END AS ctr_prev,
            CASE
                WHEN sessions_prev > 0 THEN 100.0 * engaged_prev / sessions_prev
            END AS engagement_rate_prev,
            CASE
                WHEN pos_avg_prev <= 3 THEN 'top_3'
                WHEN pos_avg_prev <= 10 THEN 'page_1'
                WHEN pos_avg_prev <= 20 THEN 'striking'
                WHEN pos_avg_prev <= 50 THEN 'page_3_5'
                ELSE 'deep'
            END AS position_tier
        FROM prev_daily
        WHERE pos_avg_prev > 0
    ),
    prev_ranked AS (
        SELECT
            *,
            MEDIAN(ctr_prev) OVER (PARTITION BY position_tier) AS tier_median_ctr
        FROM prev_scored
    ),
    next_daily AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS imp_next,
            SUM(gsc_clicks) AS clk_next,
            AVG(NULLIF(gsc_avg_position, 0)) AS pos_avg_next
        FROM {FACT_NEXT}
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 500
    ),
    next_scored AS (
        SELECT
            *,
            CASE WHEN imp_next > 0 THEN 100.0 * clk_next / imp_next END AS ctr_next,
            CASE
                WHEN pos_avg_next <= 3 THEN 'top_3'
                WHEN pos_avg_next <= 10 THEN 'page_1'
                WHEN pos_avg_next <= 20 THEN 'striking'
                WHEN pos_avg_next <= 50 THEN 'page_3_5'
                ELSE 'deep'
            END AS position_tier_next
        FROM next_daily
        WHERE pos_avg_next > 0
    ),
    next_labeled AS (
        SELECT
            *,
            CASE
                WHEN ctr_next < MEDIAN(ctr_next) OVER (PARTITION BY position_tier_next)
                     AND imp_next >= 500
                THEN 1
                ELSE 0
            END AS is_ctr_underperformer
        FROM next_scored
    )
    SELECT
        p.*,
        n.imp_next,
        n.ctr_next,
        n.is_ctr_underperformer
    FROM prev_ranked p
    INNER JOIN next_labeled n
        USING (client_hash_id, content_hash_id)
""").df()

model_df = features.dropna(subset=["ctr_prev", "pos_avg_prev"]).copy()
model_df["engagement_rate_prev"] = model_df["engagement_rate_prev"].fillna(0)
model_df["baseline_score"] = model_df["tier_median_ctr"] - model_df["ctr_prev"]

print(f"Frame n = {len(model_df):,} | clients = {model_df['client_hash_id'].nunique():,}")
print(f"Label base rate (all pages): {model_df[LABEL].mean():.3f}")

# --- Client GroupKFold ---
gkf = GroupKFold(n_splits=N_FOLDS)
groups = model_df["client_hash_id"].to_numpy()
X_all = model_df[FEATURE_COLS]
y_all = model_df[LABEL].to_numpy()

fold_rows = []
holdout_parts = []  # client-holdout scores for the ranked queue
detail = None  # last fold kept for error analysis

for fold, (tr_idx, te_idx) in enumerate(gkf.split(X_all, y_all, groups), start=1):
    train_df = model_df.iloc[tr_idx].copy()
    test_df = model_df.iloc[te_idx].copy()

    preprocessor = ColumnTransformer(
        [
            ("num", StandardScaler(), NUM_COLS),
            ("cat", OneHotEncoder(handle_unknown="ignore"), ["position_tier"]),
        ]
    )
    pipe = Pipeline(
        [
            ("prep", preprocessor),
            ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
        ]
    )
    pipe.fit(train_df[FEATURE_COLS], train_df[LABEL])
    test_df = test_df.assign(model_score=pipe.predict_proba(test_df[FEATURE_COLS])[:, 1])

    eligible = test_df[test_df["imp_prev"] >= IMP_FLOOR].copy()
    eligible["fold"] = fold
    holdout_parts.append(eligible)
    y_elig = eligible[LABEL].to_numpy()
    imp_elig = eligible["imp_prev"].to_numpy()
    base_rate = float(y_elig.mean()) if len(eligible) else float("nan")

    for method, scores in [
        ("baseline (ctr_gap)", eligible["baseline_score"].to_numpy()),
        ("logistic regression", eligible["model_score"].to_numpy()),
    ]:
        row = {
            "fold": fold,
            "method": method,
            "n_test_pages": int(len(eligible)),
            "n_test_clients": int(test_df["client_hash_id"].nunique()),
            "base_rate": base_rate,
            "tie_break": "imp_prev",
        }
        for k in KS:
            row[f"precision_at_{k}"] = precision_at_k(
                scores, y_elig, k, tie_break=imp_elig
            )
        fold_rows.append(row)

    detail = {
        "fold": fold,
        "model": pipe,
        "eligible": eligible,
        "train_clients": int(train_df["client_hash_id"].nunique()),
        "test_clients": int(test_df["client_hash_id"].nunique()),
    }

fold_df = pd.DataFrame(fold_rows)
summary = (
    fold_df.groupby("method", as_index=False)
    .agg(
        n_folds=("fold", "count"),
        n_test_pages_mean=("n_test_pages", "mean"),
        base_rate_mean=("base_rate", "mean"),
        precision_at_10=("precision_at_10", "mean"),
        precision_at_20=("precision_at_20", "mean"),
        precision_at_50=("precision_at_50", "mean"),
    )
    .set_index("method")
)

print(f"\nClient GroupKFold — {N_FOLDS} folds | tie_break=imp_prev | page-equal weight")
print("Mean Precision@K across folds (same rows within each fold):\n")
print(summary.round(3).to_string())
print("\nPer-fold detail (Precision@50):")
print(
    fold_df.pivot_table(
        index="fold",
        columns="method",
        values="precision_at_50",
    )
    .round(3)
    .to_string()
)

OUT = ROOT / "work" / "outputs" / "model_vs_baseline_metrics.json"
OUT.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "label": LABEL,
    "slice": "features=2026-03+2026-04; label=2026-05",
    "population": f"imp_prev>={IMP_FLOOR} test pages",
    "split": f"GroupKFold clients n_splits={N_FOLDS}",
    "eval_list": "one global list; page-equal weight; Precision@10/20/50",
    "tie_break": "imp_prev",
    "queue": "client-holdout scores (no all-client refit)",
    "random_state": RANDOM_STATE,
    "fold_rows": fold_df.to_dict(orient="records"),
    "methods_mean": summary.reset_index().to_dict(orient="records"),
}
OUT.write_text(json.dumps(payload, indent=2))
print(f"\nSaved → {OUT}")

# --- Ranked queue from client-holdout fold scores (time already in frame) ---
queue = pd.concat(holdout_parts, ignore_index=True)
assert len(queue) == len(model_df), "client-holdout queue should cover every eligible page once"

# Same page reason codes as w04 (past features only — not model/baseline score)
VISIBLE_TIERS = ("top_3", "page_1", "striking")
DEEP_TIERS = ("page_3_5", "deep")
ACTION_BY_REASON = {
    "zero_click_visible": "check_tracking_first",
    "high_visibility_ctr_gap": "refresh_and_review_ctr",
    "deep_tier_ctr_gap": "ranking_not_ctr",
    "general_ctr_monitor": "monitor",
}


def reason_code(row) -> str:
    tier = row["position_tier"]
    below = row["ctr_prev"] < row["tier_median_ctr"]
    if row["ctr_prev"] == 0 and tier in VISIBLE_TIERS:
        return "zero_click_visible"
    if below and tier in VISIBLE_TIERS:
        return "high_visibility_ctr_gap"
    if below and tier in DEEP_TIERS:
        return "deep_tier_ctr_gap"
    return "general_ctr_monitor"


queue["reason_code"] = queue.apply(reason_code, axis=1)
queue["action"] = queue["reason_code"].map(ACTION_BY_REASON)

queue = queue.sort_values(["model_score", "imp_prev"], ascending=[False, False])
queue["model_rank"] = np.arange(1, len(queue) + 1)
queue = queue.sort_values(["baseline_score", "imp_prev"], ascending=[False, False])
queue["baseline_rank"] = np.arange(1, len(queue) + 1)
queue = queue.sort_values("model_rank")  # model order (CSV export lives in w07)

print(f"Client-holdout ranked pages: {len(queue):,} (queue CSV exported in w07)")
print("Reason-code mix:")
print(queue["reason_code"].value_counts().to_string())

# Expose last fold for section 4
model = detail["model"]
eligible_test = detail["eligible"]
print(
    f"\nDetail fold {detail['fold']}: "
    f"train clients={detail['train_clients']}, "
    f"test clients={detail['test_clients']}, "
    f"eligible pages={len(eligible_test):,}"
)


Frame n = 52,001 | clients = 41
Label base rate (all pages): 0.515

Client GroupKFold — 4 folds | tie_break=imp_prev | page-equal weight
Mean Precision@K across folds (same rows within each fold):

                     n_folds  n_test_pages_mean  base_rate_mean  precision_at_10  precision_at_20  precision_at_50
method                                                                                                            
baseline (ctr_gap)         4           13000.25           0.513             0.85            0.738            0.775
logistic regression        4           13000.25           0.513             1.00            1.000            0.945

Per-fold detail (Precision@50):
method  baseline (ctr_gap)  logistic regression
fold                                           
1                     0.72                 0.98
2                     0.78                 0.84
3                     0.84                 0.98
4                     0.76                 0.98

Saved → c:\Users\rim

## 4. Errors and interpretation

**Did the model beat the baseline?** Yes. Mean P@10/20/50: model **1.00 / 1.00 / 0.95** vs baseline **0.85 / 0.74 / 0.78** (base rate ~0.51).

**What it leans on:** Strongest signal is low past CTR (`ctr_prev` large negative weight). Position tier matters next; impressions help a little; engagement barely moves the score.

**Where it’s wrong / different:**
- Top-10 overlap with baseline = **0/10** — same metric, different queue.
- Baseline top-50 still has **12** label-0 pages; model top-10 on this fold are all label 1.
- Biggest gaps: high-traffic underperformers the baseline buried (e.g. ~1.4M imp at baseline rank ~11k, model rank 1).

**Takeaway:** Model wins Precision@K and surfaces high-stake pages `ctr_gap` deprioritizes. Still decision-support — not proof a fix will raise clicks.


In [6]:
# --- What the model leans on (last fold) ---
prep = model.named_steps["prep"]
clf = model.named_steps["clf"]
coef = pd.Series(clf.coef_[0], index=prep.get_feature_names_out()).sort_values(
    key=np.abs, ascending=False
)
print("Top logistic-regression weights (signed, standardized inputs):")
print(coef.head(8).round(3).to_string())
print()

# --- Ranks on eligible test pages (last fold) ---
test_ranked = eligible_test.sort_values(
    ["baseline_score", "imp_prev"], ascending=[False, False]
).copy()
test_ranked["baseline_rank"] = np.arange(1, len(test_ranked) + 1)
# model rank with same tie policy
order = np.lexsort(
    (-test_ranked["imp_prev"].to_numpy(), -test_ranked["model_score"].to_numpy())
)
ranks = np.empty(len(test_ranked), dtype=int)
ranks[order] = np.arange(1, len(test_ranked) + 1)
test_ranked["model_rank"] = ranks

max_gap = test_ranked["baseline_score"].max()
n_tied_max = int((test_ranked["baseline_score"] == max_gap).sum())
print(f"Eligible test pages tied on max baseline_score ({max_gap:.4f}): {n_tied_max:,}")

top10_base = set(test_ranked.head(10)["content_hash_id"])
top10_model = set(test_ranked.nsmallest(10, "model_rank")["content_hash_id"])
print(f"Top-10 overlap (same pages in both lists): {len(top10_base & top10_model)}/10")
print()

under = test_ranked[test_ranked[LABEL] == 1].copy()
under["rank_gap"] = (under["baseline_rank"] - under["model_rank"]).abs()
print("Largest baseline vs model rank gaps (underperformers only, top 5):")
disagree_cols = [
    "content_hash_id", "imp_prev", "ctr_prev", "position_tier",
    "baseline_rank", "model_rank", "rank_gap",
]
print(
    under.nlargest(5, "rank_gap")[disagree_cols].to_string(
        index=False, float_format=lambda x: f"{x:.4f}"
    )
)
print()

baseline_top50 = test_ranked.head(50)
print(
    f"Non-underperformers in baseline top-50: "
    f"{int((baseline_top50[LABEL] == 0).sum())}"
)
print()

print("Model top-10 (eligible test) — baseline rank for each:")
model_top10 = test_ranked.nsmallest(10, "model_rank")[
    [
        "content_hash_id", "imp_prev", "ctr_prev", "position_tier",
        "baseline_rank", "model_rank", LABEL,
    ]
]
print(model_top10.to_string(index=False, float_format=lambda x: f"{x:.4f}"))


Top logistic-regression weights (signed, standardized inputs):
num__ctr_prev                 -2.120
cat__position_tier_deep       -1.328
cat__position_tier_page_1      0.735
cat__position_tier_top_3       0.223
num__pos_avg_prev             -0.220
cat__position_tier_page_3_5   -0.149
num__imp_prev                  0.128
num__engagement_rate_prev     -0.053

Eligible test pages tied on max baseline_score (0.2891): 1
Top-10 overlap (same pages in both lists): 0/10

Largest baseline vs model rank gaps (underperformers only, top 5):
         content_hash_id     imp_prev  ctr_prev position_tier  baseline_rank  model_rank  rank_gap
content_eadb33b5df496f4a 1416482.0000    0.9250         top_3          11536           1     11535
content_963de14b1f58978f  279925.0000    0.3240        page_1           6878          57      6821
content_6a809713b47bd509     629.0000    0.0000      page_3_5           2463        8609      6146
content_7fba821ce4d1123a    3047.0000    0.0328      page_3_5        